In [ ]:
!git clone https://github.com/d4-5/NLP4.git
%cd NLP4

In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd NLP4

In [ ]:
from pathlib import Path
import json
import pandas as pd
import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))
EVAL_PATH = PROJECT_ROOT / 'data/sample/lab11_eval_20.jsonl'
RESULTS_PATH = PROJECT_ROOT / 'data/sample/lab11_results.jsonl'
METRICS_PATH = PROJECT_ROOT / 'data/sample/lab11_metrics.json'

with EVAL_PATH.open(encoding='utf-8') as f:
    eval_rows = [json.loads(line) for line in f if line.strip()]

print('Eval examples:', len(eval_rows))
pd.DataFrame(eval_rows).head(3)

Eval examples: 20


,text_id,text,gold,comment
0,text_10054,"""Укрзалізниці ""надавалась правова допомога при...","{'document_id': '136-26', 'document_type': 'GE...",first relevant signal per field in reading ord...
1,text_10071,Відтак суд надав слідчому ГПУ доступ до догово...,"{'document_id': '42014000000000523', 'document...",first relevant signal per field in reading ord...
2,text_10224,Про це стало відомо з ухвали Автозаводського р...,"{'document_id': '42017171090000003', 'document...",first relevant signal per field in reading ord...


In [3]:
from src.json_schema import schema_as_pretty_json
print(schema_as_pretty_json())

{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "title": "Lab 11 document signal extraction schema",
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "document_id": {
      "type": [
        "string",
        "null"
      ],
      "minLength": 1,
      "description": "Document or case identifier without the leading number sign."
    },
    "document_type": {
      "type": [
        "string",
        "null"
      ],
      "enum": [
        "CASE_ID",
        "CONTRACT_ID",
        "ORDER_ID",
        "GENERIC_DOC_ID",
        null
      ],
      "description": "Normalized document type or null if there is no document id."
    },
    "date_iso": {
      "type": [
        "string",
        "null"
      ],
      "pattern": "^\\d{4}-\\d{2}-\\d{2}$",
      "description": "Normalized date in ISO format YYYY-MM-DD."
    },
    "date_text": {
      "type": [
        "string",
        "null"
      ],
      "minLength": 1,
      "description": "Original 

In [4]:
import pandas as pd
frame = pd.DataFrame(eval_rows)
frame[['text_id', 'comment']].head(10)


,text_id,comment
0,text_10054,first relevant signal per field in reading ord...
1,text_10071,first relevant signal per field in reading ord...
2,text_10224,first relevant signal per field in reading ord...
3,text_10345,first relevant signal per field in reading ord...
4,text_10391,first relevant signal per field in reading ord...
5,text_1078,first relevant signal per field in reading ord...
6,text_1080,first relevant signal per field in reading ord...
7,text_1723,first relevant signal per field in reading ord...
8,text_4100,first relevant signal per field in reading ord...
9,text_4209,first relevant signal per field in reading ord...


In [5]:
from src.llm_extract import build_extraction_messages
sample_prompt = build_extraction_messages(eval_rows[0]['text'])
sample_prompt


[{'role': 'system',
  'content': 'You are an information extraction engine. Return only JSON. Task=document_signal_extraction. Schema version=1.0. Return exactly one JSON object with these required keys: document_id, document_type, date_iso, date_text, amount_value, amount_currency. Use null when a value is absent. If document_id is null then document_type must also be null. If amount_value is null then amount_currency must also be null. date_iso must use YYYY-MM-DD when present. Allowed document_type values: CASE_ID, CONTRACT_ID, ORDER_ID, GENERIC_DOC_ID, null. Allowed amount_currency values: UAH, USD, EUR, UNKNOWN, null. Do not include explanations, markdown, comments, or extra keys.'},
 {'role': 'user',
  'content': 'Extract the first relevant document/date/amount signals in reading order from the Ukrainian text. If a field is absent, return null. Keep date_text exactly as in the source when present.\n\nJSON schema:\n{\n  "$schema": "https://json-schema.org/draft/2020-12/schema",\n 

In [6]:
from src.llm_extract import extract_once, DEFAULT_MODEL
sample_raw = extract_once(eval_rows[0]['text'], model=DEFAULT_MODEL)
print(sample_raw.raw_text)
print(sample_raw.validation.error_messages())


{"document_id": null, "document_type": null, "date_iso": null, "date_text": null, "amount_value": null, "amount_currency": null}
[]


In [7]:
from src.validator import validate_output
validator_demo = validate_output(sample_raw.raw_text)
{
    'parse_success': validator_demo.parse_success,
    'schema_success': validator_demo.schema_success,
    'valid': validator_demo.is_valid,
    'errors': validator_demo.error_messages(),
}


{'parse_success': True, 'schema_success': True, 'valid': True, 'errors': []}

In [8]:
from src.repair_loop import run_single_example
single_run = run_single_example(eval_rows[0], model=DEFAULT_MODEL, max_repairs=2)
{
    'raw_valid': single_run.raw_valid,
    'final_valid': single_run.final_valid,
    'repairs_used': single_run.repairs_used,
    'raw_errors': single_run.raw_errors,
    'final_errors': single_run.final_errors,
}


{'raw_valid': True,
 'final_valid': True,
 'repairs_used': 0,
 'raw_errors': [],
 'final_errors': []}

In [9]:
from src.repair_loop import run_evaluation
runs, metrics = run_evaluation(EVAL_PATH, output_path=RESULTS_PATH, model=DEFAULT_MODEL, max_repairs=2)
metrics


[lab11] processing 1/20 text_10054
[lab11] processing 2/20 text_10071
[lab11] processing 3/20 text_10224
[lab11] processing 4/20 text_10345
[lab11] processing 5/20 text_10391
[lab11] processing 6/20 text_1078
[lab11] processing 7/20 text_1080
[lab11] processing 8/20 text_1723
[lab11] processing 9/20 text_4100
[lab11] processing 10/20 text_4209
[lab11] processing 11/20 text_4225
[lab11] processing 12/20 text_527
[lab11] processing 13/20 text_8894
[lab11] processing 14/20 text_8919
[lab11] processing 15/20 text_8927
[lab11] processing 16/20 text_9076
[lab11] processing 17/20 text_9256
[lab11] processing 18/20 text_9326
[lab11] processing 19/20 text_9677
[lab11] processing 20/20 text_9843


{'total_examples': 20,
 'raw_valid_json_rate': 0.65,
 'post_repair_valid_json_rate': 0.95,
 'schema_valid_json_rate': 0.95,
 'raw_parse_failure_rate': 0.0,
 'raw_schema_failure_rate': 0.0,
 'repair_needed_rate': 0.35,
 'repair_failure_rate': 0.05,
 'average_repairs_per_example': 0.4,
 'semantic_exact_match_rate': 0.25}